# Yeast Fusion Segmenter - Google Colab Edition

This notebook provides a complete environment for:
- **Training** YOLO segmentation models on yeast fusion microscopy data
- **Prediction** on new microscopy images
- **File upload** for custom datasets and models

## Quick Start
1. Run the **Environment Setup** section first
2. Choose your workflow:
   - **Upload & Predict**: Upload images for segmentation
   - **Upload & Train**: Upload training data to retrain the model
   - **Use Example Data**: Work with pre-loaded data

## 1. Environment Setup

Install required packages and configure GPU (if available)

In [ ]:
# Check GPU availability
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    !nvidia-smi
else:
    print("Not running in Colab - some features may require adaptation")

In [ ]:
# Install required packages
!pip install -q ultralytics opencv-python-headless h5py scikit-image pillow matplotlib numpy pandas scipy tqdm

# Import core libraries
import os
import glob
import h5py
import numpy as np
import matplotlib.pyplot as plt
import cv2
import shutil
from PIL import Image, ImageSequence, ImageEnhance
from pathlib import Path
import tqdm
from ultralytics import YOLO
import pickle

print("✓ All packages installed successfully!")

## 2. File Upload & Data Management

Choose your data source:
- Upload your own microscopy images
- Upload a pre-trained model
- Use example data

In [ ]:
# Setup workspace directories
!mkdir -p datasets/train/images datasets/train/labels
!mkdir -p datasets/val/images datasets/val/labels
!mkdir -p datasets/test/images datasets/test/labels
!mkdir -p uploaded_images uploaded_models

print("✓ Workspace directories created")

In [ ]:
if IN_COLAB:
    from google.colab import files
    import io
    
    def upload_images():
        """Upload microscopy images for prediction"""
        print("Upload your microscopy images (TIF, PNG, or JPG)")
        uploaded = files.upload()
        
        uploaded_paths = []
        for filename, content in uploaded.items():
            filepath = f'uploaded_images/{filename}'
            with open(filepath, 'wb') as f:
                f.write(content)
            uploaded_paths.append(filepath)
            print(f"✓ Uploaded: {filename}")
        
        return uploaded_paths
    
    def upload_model():
        """Upload a pre-trained YOLO model"""
        print("Upload your YOLO model file (.pt)")
        uploaded = files.upload()
        
        if uploaded:
            filename = list(uploaded.keys())[0]
            filepath = f'uploaded_models/{filename}'
            with open(filepath, 'wb') as f:
                f.write(uploaded[filename])
            print(f"✓ Model uploaded: {filename}")
            return filepath
        return None
    
    def upload_training_data():
        """Upload H5 mask files and corresponding images for training"""
        print("Upload your training data (H5 masks and TIF images)")
        print("Expected format: F##_im.TIF, F##_GFP_im.TIF, F##_RFP_im.TIF, F##_mask.h5")
        uploaded = files.upload()
        
        uploaded_paths = {'images': [], 'masks': [], 'gfp': [], 'rfp': []}
        for filename, content in uploaded.items():
            filepath = f'uploaded_images/{filename}'
            with open(filepath, 'wb') as f:
                f.write(content)
            
            # Categorize uploads
            if '_mask.h5' in filename:
                uploaded_paths['masks'].append(filepath)
            elif 'GFP' in filename:
                uploaded_paths['gfp'].append(filepath)
            elif 'RFP' in filename:
                uploaded_paths['rfp'].append(filepath)
            elif '_im.TIF' in filename or '_im.tif' in filename:
                uploaded_paths['images'].append(filepath)
            
            print(f"✓ Uploaded: {filename}")
        
        return uploaded_paths
    
    print("✓ Upload functions ready")
else:
    print("Not in Colab - file upload functions disabled")
    def upload_images():
        print("Upload functions only available in Google Colab")
        return []
    def upload_model():
        print("Upload functions only available in Google Colab")
        return None
    def upload_training_data():
        print("Upload functions only available in Google Colab")
        return {}

## 3. Model Management

Load or download YOLO models

In [ ]:
def get_model(model_path=None):
    """
    Get a YOLO model - either uploaded, local, or download base model
    
    Args:
        model_path: Path to model file (optional)
    
    Returns:
        YOLO model instance
    """
    if model_path and os.path.exists(model_path):
        print(f"Loading model from: {model_path}")
        model = YOLO(model_path)
    else:
        # Download base YOLOv8 segmentation model
        print("Downloading base YOLOv8n-seg model...")
        model = YOLO("yolov8n-seg.pt")
    
    print("✓ Model loaded successfully")
    return model

# Initialize with base model
model = get_model()

## 4. Utility Functions for Image Processing

In [ ]:
import skimage.measure as measure
import copy

def output_contours(m, cl, verbose=False):
    """Extract contours from mask for YOLO format"""
    c = []
    for val in list(np.unique(m)):
        sub = copy.deepcopy(m)
        sub[sub != val] = 0
        c += measure.find_contours(sub, 0.9)
    contours = c
    
    if verbose:
        plt.imshow(m)
        plt.title(f'contours {cl}')
        for n, contour in enumerate(contours):
            plt.plot(contour[:, 1], contour[:, 0], linewidth=2)
        plt.show()
    
    lines = []
    for c in contours:
        coords = []
        for i in range(0, c.shape[0]):
            coords.append((float(c[i][1]) / m.shape[0]))
            coords.append((float(c[i][0]) / m.shape[1]))
        line = str(cl) + ' ' + ' '.join([str(c) for c in coords]) + '\n'
        lines.append(line)
    return lines

def split_mask(mask, crop=1024, nclasses=6):
    """Split multi-class mask into separate class masks"""
    mask = mask[0:crop, 0:crop]
    masks = []
    for i in range(nclasses):
        newmask = copy.deepcopy(mask)
        newmask[(newmask < (i * 1000)) | (newmask >= ((i + 1) * 1000))] = 0
        masks.append(newmask)
    return masks

def mask2contourfile(mask, outputfile, verbose=False):
    """Convert mask to YOLO format contour file"""
    if type(mask) == list:
        masks = mask
    else:
        masks = split_mask(mask)
    
    lines = None
    for i, m in enumerate(masks):
        if lines is None:
            lines = output_contours(m, i, verbose=verbose)
        else:
            lines += output_contours(m, i, verbose=verbose)
    
    with open(outputfile, 'w') as f:
        for l in lines:
            f.write(l)
    return outputfile

def yield_frames(img, crop=1024, verbose=False, scaler=True):
    """Extract frames from multi-frame TIF images"""
    for i, page in enumerate(ImageSequence.Iterator(img)):
        if verbose:
            plt.imshow(np.array(page))
            plt.show()
        if crop is not None:
            page = np.array(page)[0:crop, 0:crop]
        if scaler:
            page = (page - page.min()) / (page.max() - page.min()) * 255
        yield page

print("✓ Utility functions loaded")

## 5. Prediction Workflow

Upload and predict on new images

In [ ]:
# OPTION 1: Upload images for prediction
if IN_COLAB:
    print("Click 'Choose Files' to upload images for prediction")
    print("Supported formats: PNG, JPG, TIF")
    
    # Uncomment the line below to trigger upload
    # prediction_images = upload_images()
else:
    # For local use, specify image paths
    prediction_images = glob.glob('uploaded_images/*.png')[:5]  # Adjust path as needed

print(f"Ready to predict on {len(prediction_images) if 'prediction_images' in dir() else 0} images")

In [ ]:
from scipy.stats import describe
import pandas as pd

def predict_and_analyze(model, imgfile, outcsv=None, savefig=True, show_plot=True):
    """
    Run prediction on an image and extract quantitative features
    
    Args:
        model: YOLO model
        imgfile: Path to image file
        outcsv: Path to save CSV results (optional)
        savefig: Save prediction visualization
        show_plot: Display the plot
    
    Returns:
        DataFrame with results
    """
    # Run prediction
    results = model(imgfile, imgsz=1024, visualize=False)
    
    if len(results[0].boxes) == 0:
        print(f"No detections in {imgfile}")
        return None
    
    classes = results[0].names
    proba = results[0].boxes.conf.cpu().numpy()
    xywh = results[0].boxes.xyxy.cpu().numpy()
    
    # Load image
    img = Image.open(imgfile)
    img_array = np.array(img)
    
    # Setup plot
    plt.figure(figsize=(10, 10))
    plt.imshow(img_array)
    
    # Process each detection
    resdict = {}
    for i in range(len(proba)):
        if proba[i] > 0.5:
            x1, y1, x2, y2 = xywh[i]
            cls = int(results[0].boxes.cls[i].cpu().numpy())
            
            # Draw bounding box
            plt.plot([x1, x2, x2, x1, x1], [y1, y1, y2, y2, y1], 
                    linewidth=2, color='lime')
            
            label = f"{classes[cls]} {proba[i]:.2f}"
            plt.text(x1, y1, label, color='lime', fontsize=10,
                    bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))
            
            # Extract statistics from region
            x1_int, y1_int = int(x1), int(y1)
            x2_int, y2_int = int(x2), int(y2)
            
            # Handle multi-channel images
            if len(img_array.shape) == 3:
                region = img_array[y1_int:y2_int, x1_int:x2_int, :]
                stats_dict = {}
                
                for ch, ch_name in enumerate(['bf', 'gfp', 'rfp'][:img_array.shape[2]]):
                    ch_data = region[:, :, ch].ravel()
                    ch_stats = describe(ch_data)
                    stats_dict.update({
                        f'{ch_name}_mean': ch_stats.mean,
                        f'{ch_name}_std': np.sqrt(ch_stats.variance),
                        f'{ch_name}_min': ch_stats.minmax[0],
                        f'{ch_name}_max': ch_stats.minmax[1]
                    })
            else:
                region = img_array[y1_int:y2_int, x1_int:x2_int].ravel()
                stats = describe(region)
                stats_dict = {
                    'intensity_mean': stats.mean,
                    'intensity_std': np.sqrt(stats.variance),
                    'intensity_min': stats.minmax[0],
                    'intensity_max': stats.minmax[1]
                }
            
            resdict[i] = {
                'class': classes[cls],
                'confidence': proba[i],
                'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                'width': x2 - x1,
                'height': y2 - y1,
                'area': (x2 - x1) * (y2 - y1)
            }
            resdict[i].update(stats_dict)
    
    plt.title(f"Predictions: {len(resdict)} objects detected")
    plt.axis('off')
    
    if savefig:
        outfig = imgfile.replace('.png', '_pred.png').replace('.jpg', '_pred.png')
        plt.savefig(outfig, dpi=150, bbox_inches='tight')
        print(f"✓ Saved prediction to: {outfig}")
    
    if show_plot:
        plt.show()
    else:
        plt.close()
    
    # Create DataFrame
    if resdict:
        df = pd.DataFrame.from_dict(resdict, orient='index')
        
        if outcsv:
            df.to_csv(outcsv)
            print(f"✓ Saved results to: {outcsv}")
        
        return df
    
    return None

print("✓ Prediction function ready")

In [ ]:
# Run predictions on uploaded/specified images
if 'prediction_images' in dir() and len(prediction_images) > 0:
    print(f"Running predictions on {len(prediction_images)} images...\n")
    
    all_results = []
    for img_path in prediction_images:
        print(f"Processing: {img_path}")
        df = predict_and_analyze(
            model, img_path,
            outcsv=img_path.replace('.png', '_results.csv').replace('.jpg', '_results.csv'),
            savefig=True,
            show_plot=True
        )
        if df is not None:
            all_results.append(df)
        print("-" * 50)
    
    # Combine all results
    if all_results:
        combined_results = pd.concat(all_results, ignore_index=True)
        print("\n=== Summary Statistics ===")
        print(combined_results.describe())
        
        # Save combined results
        combined_results.to_csv('all_predictions_summary.csv', index=False)
        print("\n✓ Combined results saved to: all_predictions_summary.csv")
else:
    print("No images to predict. Upload images first or specify paths.")

## 6. Training Data Preparation

Upload and prepare training data

In [ ]:
# OPTION 2: Upload training data
if IN_COLAB:
    print("Upload training data (H5 masks + TIF images)")
    print("Expected naming: F##_im.TIF, F##_GFP_im.TIF, F##_RFP_im.TIF, F##_mask.h5")
    
    # Uncomment to trigger upload
    # training_data = upload_training_data()
else:
    # For local use - adjust paths
    print("Loading local training data...")
    training_data = {
        'images': glob.glob('images_CNN_clean/F*_im.TIF'),
        'masks': glob.glob('images_CNN_clean/F*_mask.h5'),
        'gfp': glob.glob('images_CNN_clean/*GFP_im.TIF'),
        'rfp': glob.glob('images_CNN_clean/*RFP_im.TIF')
    }

if 'training_data' in dir():
    print(f"✓ Training data loaded:")
    print(f"  - Images: {len(training_data.get('images', []))}")
    print(f"  - Masks: {len(training_data.get('masks', []))}")
    print(f"  - GFP: {len(training_data.get('gfp', []))}")
    print(f"  - RFP: {len(training_data.get('rfp', []))}")

In [ ]:
def prepare_training_dataset(training_data, crop=1024, verbose=False):
    """
    Prepare training dataset from uploaded files
    
    Args:
        training_data: Dictionary with 'images', 'masks', 'gfp', 'rfp' lists
        crop: Image crop size
        verbose: Show debug visualizations
    
    Returns:
        Number of prepared samples
    """
    # Sort and match files
    img_list = sorted(training_data.get('images', []))
    mask_list = sorted(training_data.get('masks', []))
    fluo_gfp = sorted(training_data.get('gfp', []))
    fluo_rfp = sorted(training_data.get('rfp', []))
    
    if not (img_list and mask_list and fluo_gfp and fluo_rfp):
        print("Error: Missing required files")
        return 0
    
    print(f"Processing {len(img_list)} samples...")
    
    count = 0
    for idx in tqdm.tqdm(range(min(len(img_list), len(mask_list), len(fluo_gfp), len(fluo_rfp)))):
        try:
            # Load mask
            maskfile = mask_list[idx]
            maskh5 = h5py.File(maskfile, 'r')
            
            # Get first valid mask
            mask = None
            for group in maskh5.keys():
                for frame in maskh5[group]:
                    mask = np.array(maskh5[group][frame], dtype=np.uint16)
                    if np.sum(mask) > 0:
                        mask = mask[0:crop, 0:crop]
                        break
                if mask is not None:
                    break
            
            if mask is None:
                continue
            
            # Convert mask to YOLO format
            mask_poly_file = f'/tmp/mask_{idx}.txt'
            mask2contourfile(mask, mask_poly_file, verbose=verbose)
            
            # Load and process images
            img = Image.open(img_list[idx])
            img_frames = [frame for frame in yield_frames(img, scaler=True, verbose=verbose)]
            
            gfp = Image.open(fluo_gfp[idx])
            gfp_frames = [frame for frame in yield_frames(gfp, scaler=True, verbose=verbose)]
            
            rfp = Image.open(fluo_rfp[idx])
            rfp_frames = [frame for frame in yield_frames(rfp, scaler=True, verbose=verbose)]
            
            # Stack and save frames
            for frame_idx in range(min(len(img_frames), len(gfp_frames), len(rfp_frames))):
                stacked = np.stack([
                    img_frames[frame_idx],
                    gfp_frames[frame_idx],
                    rfp_frames[frame_idx]
                ], axis=-1)
                
                # Save image
                img_outpath = f'datasets/train/images/img_{count:06d}.png'
                cv2.imwrite(img_outpath, stacked)
                
                # Copy label
                label_outpath = f'datasets/train/labels/img_{count:06d}.txt'
                shutil.copyfile(mask_poly_file, label_outpath)
                
                count += 1
        
        except Exception as e:
            print(f"Error processing sample {idx}: {e}")
            continue
    
    print(f"\n✓ Prepared {count} training samples")
    return count

# Prepare dataset if training data is available
if 'training_data' in dir() and training_data.get('images'):
    num_samples = prepare_training_dataset(training_data)
else:
    print("No training data available. Upload files first.")

In [ ]:
# Split into train/val/test sets
def split_dataset(train_ratio=0.8, val_ratio=0.1):
    """Split training data into train/val/test sets"""
    import random
    
    files = os.listdir('datasets/train/images/')
    if not files:
        print("No training data to split")
        return
    
    random.shuffle(files)
    
    n_total = len(files)
    n_val = int(n_total * val_ratio)
    n_test = int(n_total * val_ratio)
    
    # Move validation files
    val_files = files[:n_val]
    for f in val_files:
        shutil.move(f'datasets/train/images/{f}', f'datasets/val/images/{f}')
        shutil.move(f'datasets/train/labels/{f.replace(".png", ".txt")}', 
                   f'datasets/val/labels/{f.replace(".png", ".txt")}')
    
    # Move test files
    remaining_files = files[n_val:]
    test_files = remaining_files[:n_test]
    for f in test_files:
        shutil.move(f'datasets/train/images/{f}', f'datasets/test/images/{f}')
        shutil.move(f'datasets/train/labels/{f.replace(".png", ".txt")}', 
                   f'datasets/test/labels/{f.replace(".png", ".txt")}')
    
    print(f"✓ Dataset split complete:")
    print(f"  - Train: {len(os.listdir('datasets/train/images/'))} samples")
    print(f"  - Val: {len(os.listdir('datasets/val/images/'))} samples")
    print(f"  - Test: {len(os.listdir('datasets/test/images/'))} samples")

# Split if we have training data
if os.listdir('datasets/train/images/'):
    split_dataset()

## 7. Training Configuration

In [ ]:
# Create dataset YAML configuration
dataset_yaml = """
train: /content/datasets/train
val: /content/datasets/val
test: /content/datasets/test

names:
  0: f
  1: h
  2: lmcf
  3: lmsgfp
  4: lsgfp
  5: dip

# Augmentation hyperparameters
translate: 0.1
scale: 0.2
shear: 0.2
perspective: 0.1
flipud: 0.7
fliplr: 0.5
mosaic: 0.3
mixup: 0.0
copy_paste: 0.1
"""

with open('dataset.yaml', 'w') as f:
    f.write(dataset_yaml)

print("✓ Dataset configuration created")

In [ ]:
# Training hyperparameters
train_config = {
    'batch': 4,  # Adjust based on GPU memory
    'epochs': 100,  # Adjust as needed
    'imgsz': 1024,
    'device': 0,  # Use GPU 0 (Colab provides GPU)
    'lr0': 0.001,
    'lrf': 0.0001,
    'momentum': 0.5,
    'weight_decay': 0.0001,
    'warmup_epochs': 3.0,
    'patience': 20,  # Early stopping
    'save': True,
    'exist_ok': True,
    'pretrained': True,
    'optimizer': 'AdamW'
}

print("Training configuration:")
for key, val in train_config.items():
    print(f"  {key}: {val}")

## 8. Model Training

Train or fine-tune the YOLO model

In [ ]:
# Train the model
def train_model(model, config, data_yaml='dataset.yaml'):
    """
    Train YOLO segmentation model
    
    Args:
        model: YOLO model instance
        config: Training configuration dictionary
        data_yaml: Path to dataset YAML
    
    Returns:
        Training results
    """
    print("Starting training...")
    print(f"Using configuration: {config}")
    
    results = model.train(
        data=data_yaml,
        **config
    )
    
    print("\n✓ Training completed!")
    return results

# Check if we have training data before training
train_images = glob.glob('datasets/train/images/*.png')
val_images = glob.glob('datasets/val/images/*.png')

if len(train_images) > 0 and len(val_images) > 0:
    print(f"Ready to train on {len(train_images)} training samples")
    print(f"Validation set: {len(val_images)} samples")
    
    # Uncomment to start training
    # results = train_model(model, train_config)
    # model.save('yolov8n-seg_yeast_fusion_trained.pt')
    # print("Model saved as: yolov8n-seg_yeast_fusion_trained.pt")
else:
    print("Not enough training data. Upload and prepare training data first.")

In [ ]:
# Optional: Resume training from checkpoint
def resume_training(checkpoint_path, config):
    """Resume training from a checkpoint"""
    model = YOLO(checkpoint_path)
    results = model.train(resume=True, **config)
    return results

# Uncomment to resume training
# results = resume_training('runs/segment/train/weights/last.pt', train_config)

## 9. Model Evaluation

In [ ]:
# Evaluate trained model
def evaluate_model(model, data_yaml='dataset.yaml'):
    """Evaluate model on validation/test set"""
    print("Evaluating model...")
    
    results = model.val(data=data_yaml)
    
    print("\n=== Evaluation Results ===")
    print(f"mAP50: {results.box.map50:.3f}")
    print(f"mAP50-95: {results.box.map:.3f}")
    
    # Display confusion matrix if available
    if hasattr(results, 'confusion_matrix'):
        print("\nConfusion matrix:")
        print(results.confusion_matrix)
    
    return results

# Evaluate if model is trained
if os.path.exists('runs/segment/train/weights/best.pt'):
    trained_model = YOLO('runs/segment/train/weights/best.pt')
    eval_results = evaluate_model(trained_model)

## 10. Download Results

Download trained models and predictions

In [ ]:
if IN_COLAB:
    from google.colab import files
    import zipfile
    
    def download_results():
        """Package and download results"""
        print("Preparing results for download...")
        
        # Create zip of results
        results_files = []
        
        # Add trained model if exists
        if os.path.exists('runs/segment/train/weights/best.pt'):
            results_files.append('runs/segment/train/weights/best.pt')
        
        # Add prediction results
        results_files.extend(glob.glob('uploaded_images/*_pred.png'))
        results_files.extend(glob.glob('uploaded_images/*_results.csv'))
        results_files.append('all_predictions_summary.csv')
        
        if results_files:
            with zipfile.ZipFile('yeast_fusion_results.zip', 'w') as zipf:
                for file in results_files:
                    if os.path.exists(file):
                        zipf.write(file)
            
            print("✓ Results packaged")
            files.download('yeast_fusion_results.zip')
        else:
            print("No results to download")
    
    # Uncomment to download results
    # download_results()
else:
    print("Download function only available in Google Colab")

## 11. Complete Workflow Examples

Ready-to-run examples for common tasks

In [ ]:
# WORKFLOW 1: Quick Prediction
# 1. Upload images: prediction_images = upload_images()
# 2. Run: predict_and_analyze(model, prediction_images[0], show_plot=True)

# WORKFLOW 2: Train from Scratch
# 1. Upload training data: training_data = upload_training_data()
# 2. Prepare: prepare_training_dataset(training_data)
# 3. Split: split_dataset()
# 4. Train: results = train_model(model, train_config)
# 5. Save: model.save('my_trained_model.pt')

# WORKFLOW 3: Fine-tune Existing Model
# 1. Upload model: custom_model_path = upload_model()
# 2. Load: model = get_model(custom_model_path)
# 3. Upload data and prepare as above
# 4. Train with lower learning rate: train_config['lr0'] = 0.0001
# 5. Train: results = train_model(model, train_config)

print("✓ Workflow examples ready")
print("\nUncomment the relevant code sections above to execute workflows")

## 12. Troubleshooting & Tips

### Common Issues

**Out of Memory Error:**
- Reduce `batch` size in `train_config`
- Reduce `imgsz` to 512 or 768
- Use Google Colab Pro for more GPU memory

**No GPU Available:**
- Check GPU runtime: Runtime → Change runtime type → GPU
- Free tier has limited GPU hours

**Upload Fails:**
- Check file formats (TIF, H5, PNG supported)
- Ensure filenames match expected patterns
- Try uploading in smaller batches

**Training Too Slow:**
- Reduce number of epochs
- Increase batch size (if GPU allows)
- Use smaller image size

### Best Practices

1. **Start Small:** Test with a few images before full dataset
2. **Monitor Training:** Watch loss curves for overfitting
3. **Save Frequently:** Models are saved during training
4. **Validate Early:** Check predictions after few epochs
5. **Use Augmentation:** Helps with small datasets

### Data Format

**Training data should include:**
- Brightfield images: `F##_im.TIF`
- GFP fluorescence: `F##_GFP_im.TIF`
- RFP fluorescence: `F##_RFP_im.TIF`
- Segmentation masks: `F##_mask.h5`

**Where ## is a zero-padded number (01, 02, etc.)**

---
## Summary

This notebook provides:
- ✅ Complete environment setup for Google Colab
- ✅ File upload capabilities for images and models
- ✅ Prediction pipeline with quantitative analysis
- ✅ Training pipeline from custom data
- ✅ Model evaluation and export

**Next Steps:**
1. Choose your workflow (prediction or training)
2. Upload your data
3. Run the corresponding cells
4. Download your results

For questions or issues, refer to the troubleshooting section above.